# 04. 파이썬 기초 - pandas 데이터분석

`pandas` 는 표(엑셀 같은) 데이터를 다루는 핵심 라이브러리입니다.
이 프로젝트의 넷플릭스·K-Pop 분석이 모두 pandas 로 작성되어 있습니다.

**다루는 내용**
1. Series 와 DataFrame
2. '딕셔너리들의 리스트' → DataFrame
3. 열 선택 / 행 필터링 (loc)
4. 정렬, value_counts
5. 결측치 처리
6. groupby 집계
7. 날짜 변환 (to_datetime)

In [8]:
import pandas as pb   # 관례적으로 pd 라는 별칭 사용
print('pandas 버전:', pb.__version__)

pandas 버전: 2.2.2


## 1. Series 와 DataFrame

- **Series**: 1차원 (한 개의 열)
- **DataFrame**: 2차원 표 (여러 열)

In [9]:
# Series: 리스트에 인덱스가 붙은 형태
prices = pb.Series([25000, 30000, 28000])
print(prices)
print('평균:', prices.mean())

0    25000
1    30000
2    28000
dtype: int64
평균: 27666.666666666668


## 1-1. `raise` — 내가 직접 예외를 일으키기 

지금까지는 파이썬이 **자동으로 발생시킨** 예외를 `try/except` 로 **받았습니다.**
`raise` 는 그 반대로, **내가 직접 예외를 만들어 던지는** 것입니다.

```python
raise ValueError('나이는 숫자여야 합니다')
#     └─ 예외 종류 ─┘ └────── 메시지 ──────┘
```

### 왜 필요한가

잘못된 값을 `None` 으로 돌려주고 넘어가면, 오류가 **한참 뒤 엉뚱한 곳에서** 터집니다.
`raise` 는 **문제를 발견한 그 자리에서 즉시** 알려 원인 추적을 쉽게 만듭니다.

> **핵심: `raise` 는 "에러를 내는 것" 이 아니라 "호출한 쪽에 문제를 알리는 것" 입니다.**
> 함수가 약속(계약)을 지킬 수 없을 때 조용히 넘어가지 않고 명확히 신고하는 장치입니다.

### 자주 쓰는 내장 예외

| 예외 | 언제 던지나 | 예 |
|---|---|---|
| `ValueError` | 타입은 맞는데 **값**이 잘못됨 | 나이가 `-5`, 빈 검색어 |
| `TypeError` | **타입** 자체가 잘못됨 | 숫자 자리에 문자열 |
| `KeyError` | 딕셔너리에 **키가 없음** | 없는 사용자 id |
| `FileNotFoundError` | 파일이 없음 | 설정 파일 누락 |
| `RuntimeError` | 위에 해당 없는 실행 중 오류 | 설정 로딩 실패 |

In [59]:
# =========================================================
# raise - 내가 직접 예외를 일으키기
#   목적: 잘못된 값을 '발견한 그 자리에서 즉시' 알리는 것
# =========================================================

# 방법 1) 잘못되면 None 을 돌려주기 → 문제를 뒤로 미룬다
def get_age_bad(value):
    """나이를 정수로 변환한다. 실패하면 None 을 돌려준다."""
    try:
        return int(value)
    except ValueError:
        return None


# 방법 2) raise 로 즉시 알리기 → 문제가 생긴 자리에서 드러난다
def get_age(value):
    """나이 문자열을 정수로 변환한다.

    Args:
        value (str): 나이 문자열

    Returns:
        int: 0~150 사이의 나이

    Raises:
        ValueError: 숫자가 아니거나 범위를 벗어난 경우
    """
    try:
        age = int(value)
    except ValueError:
        # raise 뒤에 '예외 객체' 를 만들어 던진다. 메시지는 원인이 보이게 쓴다
        raise ValueError(f'나이는 숫자여야 합니다: {value!r}')

    # 형식은 맞지만 값이 이상한 경우도 직접 걸러낸다
    if not 0 <= age <= 150:
        raise ValueError(f'나이 범위를 벗어났습니다: {age}')
    return age


print('정상 동작:', get_age('10'))

정상 동작: 10


In [ ]:
print()
# None 을 돌려주면 오류가 '한참 뒤 엉뚱한 곳' 에서 터진다
bad = get_age_bad('스물다섯')
print('[None 반환 방식] 반환값:', bad)
try:
    print('  내년 나이 계산:', bad + 1)
except TypeError as exp:
    print('  → 뒤늦게 TypeError:', exp)
    print('  → 진짜 원인(잘못된 입력)이 어디였는지 알 수 없다')

In [60]:

print()
# raise 방식은 원인과 위치를 그 자리에서 알려준다
for v in ['스물다섯', '200']:
    try:
        get_age(v)
    except ValueError as exp:
        print(f'[raise 방식] {v!r} → ValueError: {exp}')


[raise 방식] '스물다섯' → ValueError: 나이는 숫자여야 합니다: '스물다섯'
[raise 방식] '200' → ValueError: 나이 범위를 벗어났습니다: 200


In [ ]:
# =========================================================
# 상황에 맞는 '예외 종류' 고르기
#   메시지만큼 어떤 종류의 예외를 던지느냐도 중요하다.
#   호출하는 쪽이 except 로 골라 잡아 다르게 대응할 수 있기 때문이다.
# =========================================================
def find_user(users, user_id):
    """사용자를 찾아 이름을 돌려준다.

    Args:
        users (dict): {id: 이름} 형태의 사용자 목록
        user_id (int): 찾을 사용자 id

    Returns:
        str: 사용자 이름

    Raises:
        TypeError: user_id 가 정수가 아닌 경우 (타입 자체가 잘못됨)
        KeyError: 해당 id 의 사용자가 없는 경우 (키가 없음)
    """
    if not isinstance(user_id, int):
        raise TypeError(f'user_id 는 정수여야 합니다 (받은 타입: {type(user_id).__name__})')
    if user_id not in users:
        raise KeyError(f'존재하지 않는 사용자 id: {user_id}')
    return users[user_id]


users = {1: '홍길동', 2: '김철수'}

# 호출하는 쪽은 '예외 종류' 로 구분해 각각 다르게 대응한다
for arg in [1, '1', 99]:
    try:
        print(f'  {arg!r:>5} → {find_user(users, arg)}')
    except TypeError as exp:
        print(f'  {arg!r:>5} → [입력 형식 오류] {exp}')     # 코드를 고쳐야 하는 문제
    except KeyError as exp:
        print(f'  {arg!r:>5} → [데이터 없음] {exp}')        # 데이터 문제

# 참고: KeyError 는 메시지를 출력할 때 따옴표가 함께 붙는다 (KeyError 의 특성)

### 예외를 다시 던지거나, 바꿔서 던지기

| 문법 | 하는 일 | 언제 |
|---|---|---|
| `raise` (인자 없이) | 방금 잡은 예외를 **그대로** 다시 던짐 | 로그만 남기고 처리는 호출한 쪽에 맡길 때 |
| `raise 새예외 from 원본` | **다른 예외로 바꾸되 원인을 연결** | 저수준 예외를 우리 용어로 감쌀 때 |

`from` 을 쓰면 원본 예외가 `__cause__` 에 보존되고,
traceback 에 *"The above exception was the direct cause of the following exception"* 으로 함께 표시됩니다.

> `except` 안에서 `raise` 를 쓸 때 **`raise exp` 라고 쓰지 마세요.**
> 인자 없이 `raise` 만 쓰면 원래 traceback 이 그대로 보존되지만,
> `raise exp` 는 그 위치에서 다시 시작된 것처럼 기록되어 추적이 어려워집니다.

In [ ]:
# =========================================================
# 1) raise 단독 - 잡았던 예외를 '그대로' 다시 던지기
#    로그만 남기고, 실제 처리는 호출한 쪽에 맡길 때 쓴다.
# =========================================================
def load_config(path):
    """설정 파일을 읽는다. 없으면 로그를 남기고 예외를 다시 던진다.

    Raises:
        FileNotFoundError: 파일이 없는 경우 (그대로 다시 던짐)
    """
    try:
        with open(path, encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f'  [로그] 설정 파일 없음: {path}')
        raise      # 인자 없이 raise → 방금 잡은 예외를 그대로 다시 던진다


print('1) raise 단독')
try:
    load_config('없는파일.txt')
except FileNotFoundError as exp:
    print('  호출한 쪽에서 처리:', exp.strerror)



In [ ]:

# =========================================================
# 2) raise ... from exp - 원인을 연결해 '다른 예외로 바꿔' 던지기
#    저수준 예외(FileNotFoundError)를 우리 프로젝트 용어의 예외로 감쌀 때 쓴다.
# =========================================================
def load_settings(path):
    """설정을 읽되, 실패하면 RuntimeError 로 바꿔 던진다.

    Raises:
        RuntimeError: 설정을 불러오지 못한 경우 (원인은 __cause__ 에 보존)
    """
    try:
        with open(path, encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError as exp:
        # from exp : 원래 원인을 잃지 않고 연결해 둔다
        raise RuntimeError('설정을 불러오지 못했습니다') from exp


print('2) raise ... from')
try:
    load_settings('없는파일.txt')
except RuntimeError as exp:
    print('  겉으로 드러난 예외 :', type(exp).__name__, '-', exp)
    print('  실제 원인(__cause__):', type(exp.__cause__).__name__, '-', exp.__cause__)

print()
print('→ from 을 생략하면 원인이 끊겨 "왜 실패했는지" 를 추적할 수 없다')

In [ ]:
# =========================================================
# 사용자 정의 예외 - 내 프로젝트만의 오류 종류 만들기
#   Exception 을 상속하면 끝이다. 본문은 docstring 하나면 충분하다.
#   장점: 이름 자체가 문서가 되고, 호출하는 쪽이 정확히 골라 잡을 수 있다.
# =========================================================

class ScrapingError(Exception):
    """스크래핑 과정에서 발생하는 오류의 최상위 클래스."""


class PageNotFoundError(ScrapingError):
    """요청한 페이지가 없을 때."""


class ParseError(ScrapingError):
    """HTML 구조가 예상과 달라 파싱에 실패했을 때."""


def scrape(status_code, html):
    """응답을 검사하고 <title> 내용을 파싱한다.

    Args:
        status_code (int): HTTP 응답 코드
        html (str): 응답 본문

    Returns:
        str: title 태그 안의 텍스트

    Raises:
        PageNotFoundError: 404 응답인 경우
        ParseError: title 태그를 찾지 못한 경우
    """
    if status_code == 404:
        raise PageNotFoundError(f'페이지 없음 (status={status_code})')
    if '<title>' not in html:
        raise ParseError('title 태그를 찾을 수 없습니다')
    return html.split('<title>')[1].split('</title>')[0]


cases = [
    (200, '<html><title>파이썬</title></html>'),   # 정상
    (404, ''),                                     # 페이지 없음
    (200, '<html>제목 없음</html>'),                # 구조가 다름
]

print('예외 종류별로 다르게 대응하기:')
for code, html in cases:
    try:
        print('  성공:', scrape(code, html))
    except PageNotFoundError as exp:
        print('  건너뜀:', exp)          # 이 페이지만 넘어가고 계속 진행
    except ParseError as exp:
        print('  파싱 실패:', exp)        # 셀렉터 점검이 필요한 상황

print()
# 상속 관계 덕분에 상위 클래스 하나로 전부 잡을 수도 있다
print('ScrapingError 하나로 모두 잡기:')
for code, html in cases:
    try:
        scrape(code, html)
    except ScrapingError as exp:
        print(f'  {type(exp).__name__}: {exp}')

## 2. '딕셔너리들의 리스트' → DataFrame 

01편에서 배운 구조가 그대로 표로 변환됩니다.
스크래핑 결과 `pd.DataFrame(books)` 가 바로 이것입니다.

In [10]:
books = [
    {'title': '파이썬 입문', 'author': '홍길동', 'price': 25000, 'publisher': 'A출판'},
    {'title': '데이터 분석', 'author': '김철수', 'price': 30000, 'publisher': 'B출판'},
    {'title': '웹 스크래핑', 'author': '이영희', 'price': 28000, 'publisher': 'A출판'},
    {'title': '머신러닝',   'author': '박민수', 'price': 33000, 'publisher': 'C출판'},
]

df = pd.DataFrame(books)
df

,title,author,price,publisher
0,파이썬 입문,홍길동,25000,A출판
1,데이터 분석,김철수,30000,B출판
2,웹 스크래핑,이영희,28000,A출판
3,머신러닝,박민수,33000,C출판


In [19]:
# 데이터 훑어보기 (분석 시작 시 항상 확인하는 것들)
print('행 x 열:', df.shape)      # (4, 4)
print('\n컬럼:', list(df.columns))
print('\n--- 앞부분 미리보기 ---')
print(df.head(2))                # 위에서 2줄
print('\n--- 요약 통계 ---')
print(df['price'].describe())    # 숫자열 통계
df['price'].describe()

행 x 열: (4, 4)

컬럼: ['title', 'author', 'price', 'publisher']

--- 앞부분 미리보기 ---
    title author  price publisher
0  파이썬 입문    홍길동  25000       A출판
1  데이터 분석    김철수  30000       B출판

--- 요약 통계 ---
count        4.000000
mean     29000.000000
std       3366.501646
min      25000.000000
25%      27250.000000
50%      29000.000000
75%      30750.000000
max      33000.000000
Name: price, dtype: float64


count        4.000000
mean     29000.000000
std       3366.501646
min      25000.000000
25%      27250.000000
50%      29000.000000
75%      30750.000000
max      33000.000000
Name: price, dtype: float64

## 3. 열 선택 / 행 필터링

- 열 선택: `df['컬럼']` 또는 `df[['컬럼1','컬럼2']]`
- 행 필터: `df[조건]` — 조건이 True인 행만 남김

In [20]:
# 한 개 열 선택 → Series
print(df['title'])

print('-' * 30)

# 여러 열 선택 → DataFrame (대괄호 두 겹)
df[['title', 'price']]

0    파이썬 입문
1    데이터 분석
2    웹 스크래핑
3      머신러닝
Name: title, dtype: object
------------------------------


,title,price
0,파이썬 입문,25000
1,데이터 분석,30000
2,웹 스크래핑,28000
3,머신러닝,33000


In [26]:
# 조건 필터링: 가격이 28000 이상인 책
condition = df['price'] >= 28000
print(condition)               # True/False 의 Series
print('-' * 30)
df[condition]           # True인 행만


0    False
1     True
2     True
3     True
Name: price, dtype: bool
------------------------------


,title,author,price,publisher
1,데이터 분석,김철수,30000,B출판
2,웹 스크래핑,이영희,28000,A출판
3,머신러닝,박민수,33000,C출판


In [28]:

# loc: [행조건, 열목록] 을 한 번에 (실제 분석 코드에서 자주 사용)
df.loc[df['price'] >= 28000, ['title', 'price']]

,title,price
1,데이터 분석,30000
2,웹 스크래핑,28000
3,머신러닝,33000


## 4. 정렬과 개수 세기

In [32]:
# 가격 높은 순 정렬
print(df.sort_values('price', ascending=False))

print('-' * 30)

# value_counts(): 값별 개수 (출판사별 도서 수) — 분석에서 매우 자주 사용
df['publisher'].value_counts()

    title author  price publisher
3    머신러닝    박민수  33000       C출판
1  데이터 분석    김철수  30000       B출판
2  웹 스크래핑    이영희  28000       A출판
0  파이썬 입문    홍길동  25000       A출판
------------------------------


publisher
A출판    2
B출판    1
C출판    1
Name: count, dtype: int64

## 5. 결측치(빠진 값) 처리

실제 데이터에는 빈 값(NaN)이 흔합니다. 넷플릭스·K-Pop 데이터도 결측이 많았습니다.

In [35]:
import numpy as np

# 일부러 결측치가 있는 데이터 생성
df2 = pd.DataFrame({
    'name': ['A', 'B', 'C', 'D'],
    'height': [172, np.nan, 168, 180],   # B의 키가 결측
})
df2

,name,height
0,A,172.0
1,B,NaN
2,C,168.0
3,D,180.0


In [36]:
print('결측치 개수:\n', df2.isna().sum())     # 열별 결측 개수
print('\n결측 행 제거:\n', df2.dropna())        # NaN 있는 행 삭제
print('\n결측을 평균으로 채움:\n', df2.fillna(df2['height'].mean()))

결측치 개수:
 name      0
height    1
dtype: int64

결측 행 제거:
   name  height
0    A   172.0
2    C   168.0
3    D   180.0

결측을 평균으로 채움:
   name      height
0    A  172.000000
1    B  173.333333
2    C  168.000000
3    D  180.000000


## 6. groupby — 그룹별 집계 ⭐

'출판사별 평균 가격' 처럼 **그룹으로 묶어 계산** 합니다.
K-Pop 분석의 '소속사별 평균 데뷔 나이' 가 이 문법입니다.

In [42]:
# 출판사별 평균 가격
print(df.groupby('publisher')['price'].mean())
df

publisher
A출판    26500.0
B출판    30000.0
C출판    33000.0
Name: price, dtype: float64


,title,author,price,publisher
0,파이썬 입문,홍길동,25000,A출판
1,데이터 분석,김철수,30000,B출판
2,웹 스크래핑,이영희,28000,A출판
3,머신러닝,박민수,33000,C출판


In [40]:
# 출판사별 도서 수 + 평균 가격 (여러 집계 한 번에)
summary = df.groupby('publisher')['price'].agg(['count', 'mean'])
print(summary)

           count     mean
publisher                
A출판            2  26500.0
B출판            1  30000.0
C출판            1  33000.0


## 7. 날짜 다루기 — to_datetime

문자열로 된 날짜를 날짜형으로 바꾸면 연/월 추출, 기간 계산이 쉬워집니다.
K-Pop 분석에서 데뷔 나이를 계산할 때 사용했습니다.

In [47]:
df3 = pd.DataFrame({'debut': ['26/08/2014', '31/10/2015', '11/10/2017']})
print(df3['debut'].dtypes)
df3

object


,debut
0,26/08/2014
1,31/10/2015
2,11/10/2017


In [51]:
# 문자열 → 날짜형 (dd/mm/yyyy 형식이므로 dayfirst=True)
df3['debut'] = pd.to_datetime(df3['debut'], dayfirst=True)
print(df3['debut'].dtype)

datetime64[ns]


In [54]:

# 날짜형이 되면 .dt 로 연/월/일 추출 가능
df3['year'] = df3['debut'].dt.year
print(df3['year'].dtype)
df3

int32


,debut,year
0,2014-08-26,2014
1,2015-10-31,2015
2,2017-10-11,2017


In [56]:
# 두 날짜의 차이 (일수) 계산
gap = pd.to_datetime('2020-01-01') - pd.to_datetime('2014-08-26')
print('\n기간(일):', gap.days)


기간(일): 1954


## 정리
- **DataFrame**: '딕셔너리들의 리스트' → 표
- **훑어보기**: `shape`, `head()`, `columns`, `describe()`
- **선택/필터**: `df['col']`, `df.loc[조건, 열목록]`
- **정렬/집계**: `sort_values`, `value_counts`, `groupby().agg()`
- **결측치**: `isna()`, `dropna()`, `fillna()`
- **날짜**: `to_datetime()`, `.dt.year`

다음: `05python_basic_웹요청과파싱.ipynb` (requests·BeautifulSoup)